<a href="https://colab.research.google.com/github/Muqtarali/Generative-AI-using-LangChain/blob/main/Project/YouTube_Chatbot_using_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.environ["GROQ_API_KEY"]='gsk............'

In [ ]:
%pip install -U langchain langchain-community langchain-groq langchain-classic langchain-text-splitters langgraph faiss-cpu tiktoken youtube-transcript-api python-dotenv

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq.chat_models import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [ ]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi.fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")


## Step 1b - Indexing (Text Splitting)

In [ ]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunk = splitter.split_documents(transcript)



In [ ]:
len(chunk)

176

In [ ]:
chunk[0]

Document(metadata={'video_id': 'Gfr50f6ZBvo'}, page_content="Introduction\n0:00\nthe following is a conversation with demus hasabis ceo and co-founder of deepmind\n0:06\na company that has published and builds some of the most incredible artificial intelligence systems in the history of\n0:13\ncomputing including alfred zero that learned all by itself to play the game of gold\n0:18\nbetter than any human in the world and alpha fold two that solved protein\n0:24\nfolding both tasks considered nearly impossible for a very long time\n0:31\ndemus is widely considered to be one of the most brilliant and impactful humans in the history of artificial\n0:37\nintelligence and science and engineering in general this was truly an honor and a pleasure\n0:43\nfor me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future\n0:51\nthis is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends\n0:5

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
!pip install langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
vector_store=FAISS.from_documents(
    documents=chunk,
    embedding=HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
vector_store.index_to_docstore_id

{0: '6b401766-7f8b-4f29-867f-c41210316c37',
 1: 'f6cac9a7-1282-4d09-b288-df49b89845f5',
 2: '0a973d1d-d27a-411a-bdf3-6b9870831220',
 3: 'af3cf928-f20c-41a6-b8ef-2390976ff72a',
 4: '0a6ac2ea-f715-4417-88bc-4a8144817b49',
 5: '827a5179-6800-4681-827d-58073adfedb8',
 6: '6691e515-3faa-4286-a7f2-8700e6ab3b4c',
 7: '3321d3c9-adb7-4e6b-a2e9-4c3f99045134',
 8: '86846e6e-0473-4157-81b2-e2a46127c0de',
 9: 'ea3762bc-7685-4972-ab18-ec5c1332215b',
 10: 'fa55c482-13d0-479b-8566-46d607c71f6f',
 11: '1f2effa0-affd-4eb8-a0f3-853360dd3520',
 12: 'ed3e847b-2b6b-498f-95c0-417bcc6b2b1f',
 13: '386e2a6d-3549-4fb8-82ce-4edf70ef6a21',
 14: '55d13882-a090-46d0-98e6-5201d3dc1e99',
 15: '23f691d1-c7cc-43a5-8cd5-2892628d9197',
 16: 'e3933169-564b-4396-979f-d46b51ed1296',
 17: 'bebb60a6-78c9-4f1c-96fc-58f27b33f16b',
 18: '4765ac13-701d-4264-9f2f-ae322d766252',
 19: 'd53e8a8e-4b4c-45ae-a50d-4b4a11df2ac7',
 20: 'b7fbf1af-2c5a-41d6-a6bf-6743c2cf68e1',
 21: '451b9129-9ceb-44d3-8e78-ce0af4ab469e',
 22: 'b8722027-2c19-

In [ ]:
vector_store.get_by_ids(['c007bb30-661b-4ea3-a2f8-006993907136'])

[]

## Step 2 - Retrieval

In [ ]:
Retrieval=vector_store.as_retriever(
    type="similarity",
    search_kwargs={"k":4}
)

In [ ]:
Retrieval

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7fe71dfdd6d0>, search_kwargs={'k': 4})

In [ ]:
Retrieval.invoke('what is deepmind')

[Document(id='4f719cc6-1f54-4c83-a5ad-005d04bd6ea4', metadata={'video_id': 'Gfr50f6ZBvo'}, page_content="57:37\ni think you can use neuroscience for is is you know any at mathematics you when you're when you're doing something as\n57:42\nambitious as trying to solve intelligence and you're you're you know it's blue sky research no one knows how to do it you you you need to use any\n57:49\nevidence or any source of information you can to help guide you in the right direction or give you confidence you're\n57:55\ngoing in the right direction so so that that was one reason we pushed so hard on that and that's and just going back to\n58:01\nyour early question about organization the other big thing that i think we innovated with at deepmind to encourage\n58:07\ninvention and and uh and innovation was the multi-disciplinary organization we\n58:12\nbuilt and we still have today so deepmind originally was a confluence of the of the most cutting-edge knowledge\n58:18\nin neuroscience with mach

## Step 3 - Augmentation

In [ ]:
prompt = PromptTemplate(
    template="""You are a helpful assistant.
Answer ONLY from the provided transcript context.
If the context is insufficient, just say you don't know.

Context:
{context}

Question: {question}""",
    input_variables=["context", "question"]
)

In [ ]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = Retrieval.invoke(question)

In [ ]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
context_text

"1:12:57\nof those things in material science like room temperature superconductors or something on my list one day i'd like to\n1:13:02\nlike you know have an ai system to help build better optimized batteries all of these sort of mechanical things mr i\n1:13:08\nthink a systematic sort of search could be uh guided by a model could be um could be\n1:13:15\nextremely powerful so speaking of which you have a paper on nuclear fusion\nNuclear fusion\n1:13:20\nuh magnetic control of tokamak plasmas to deep reinforcement learning so you uh\n1:13:26\nyou're seeking to solve nuclear fusion with deep rl so it's doing control of high temperature plasmas can you explain this\n1:13:32\nwork and uh can ai eventually solve nuclear fusion it's been very fun last year or two and\n1:13:39\nvery productive because we've been taking off a lot of my dream projects if you like of things that i've collected over the years of\n1:13:45\n\n1:14:21\nwe try to do whenever we go into a new field to apply our sys

In [ ]:
final_prompt = prompt.invoke({"Context": context_text, "Question": question})

In [ ]:
final_prompt

StringPromptValue(text="You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      1:12:57\nof those things in material science like room temperature superconductors or something on my list one day i'd like to\n1:13:02\nlike you know have an ai system to help build better optimized batteries all of these sort of mechanical things mr i\n1:13:08\nthink a systematic sort of search could be uh guided by a model could be um could be\n1:13:15\nextremely powerful so speaking of which you have a paper on nuclear fusion\nNuclear fusion\n1:13:20\nuh magnetic control of tokamak plasmas to deep reinforcement learning so you uh\n1:13:26\nyou're seeking to solve nuclear fusion with deep rl so it's doing control of high temperature plasmas can you explain this\n1:13:32\nwork and uh can ai eventually solve nuclear fusion it's been very fun last year or two and\n1:13:39\nvery productive because we've bee

## Step 4 - Generation

In [ ]:
model = ChatGroq(
    model='openai/gpt-oss-20b',
    temperature=0.2,
)

In [ ]:
result=model.invoke(final_prompt)

In [ ]:
print(result.content)

Yes.  
The transcript discusses using deep reinforcement learning to control high‑temperature plasmas in a tokamak fusion reactor. It covers:

* The goal of solving nuclear fusion by learning to shape and hold the plasma using magnetic fields.  
* Collaboration with EPFL in Switzerland, which provided a test reactor and a plasma simulator.  
* Identification of bottleneck problems in fusion and selecting those that AI can address.  
* Development of a controller that can maintain various plasma shapes (e.g., “droplets”) for the required duration.  
* The broader context of applying AI to other fusion challenges and working with fusion startups.


## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def formate_docs(retrieved_docs):
  context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain=RunnableParallel({
    'context':Retrieval| RunnableLambda(formate_docs),
    'question':RunnablePassthrough()
}
)

In [ ]:
parallel_chain.invoke('who is Demis')

{'context': "2:09:44\nconsciousness uh life and gravity all of these things yeah giving us a glimpses of\n2:09:50\nexplanations for those things yeah well um damas dear one of the special\n2:09:56\nhuman beings in this giant puzzle of ours and it's a huge honor that you would take a pause from the bigger\n2:10:01\npuzzle to solve this small puzzle of a conversation with me today it's truly an honor and a pleasure thank you thank you\n2:10:06\ni really enjoyed it thanks lex thanks for listening to this conversation with demas establish to\n2:10:11\nsupport this podcast please check out our sponsors in the description and now let me leave you with some words\n2:10:17\nfrom edskar dykstra computer science is no more about computers\n2:10:23\nthan astronomy is about telescopes thank you for listening and hope to see\n2:10:28\nyou next time\n\nIntroduction\n0:00\nthe following is a conversation with demus hasabis ceo and co-founder of deepmind\n0:06\na company that has published and builds 

In [ ]:
parser=StrOutputParser()

In [ ]:
main_chain=parallel_chain|prompt|model|parser

In [ ]:
main_chain = parallel_chain | prompt | model | parser
output = main_chain.invoke("can you summaries the video")
print(output)

**Video Summary**

The video discusses how advanced computational methods—particularly machine‑learning and AI—are being used to tackle some of the most complex problems in physics and biology.

1. **Quantum‑mechanical simulations**  
   - Researchers are developing *functionals* that approximate Schrödinger’s equation, enabling them to predict how electrons behave in materials.  
   - The goal is to create a functional that closely reproduces full quantum simulations while being computationally cheaper.  
   - This involves running many molecular‑dynamics simulations on large compute clusters to generate training data, then learning the mapping from initial conditions to the desired functional.

2. **AI as a language for biology**  
   - Biology is described as messy, emergent, and highly dynamic, making it unlikely that a single elegant law (like Newton’s) will ever fully capture cellular behavior.  
   - AI can serve as a “description language” by starting from basic molecular build